<a href="https://colab.research.google.com/github/hiiamlars/master_thesis/blob/main/llm_reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [27]:
# @title Dependencies

# Installation
! pip install pandas pyarrow -q
! pip install numpy -q
! pip install openai -q

# first-party installations
import itertools

# third-party installations
from google.colab import drive
import pandas as pd
import numpy as np
from openai import OpenAI
from openai import APIError

In [ ]:
# @title Paths

# mount GDrive
drive.mount('/content/drive')

# define dataset directory
DATASET_DIR = "/content/drive/MyDrive/Thesis/dataset/"

# define target column
TARGET_COL = "parsed_text"

In [16]:
# @title Load data

# read dataset_paper
dataset_paper = pd.read_parquet(DATASET_DIR + "dataset_paper.parquet")

# select columns 'paper_id', 'abstract' and 'parsed_text'
paper_content = dataset_paper[['paper_id', 'abstract', 'parsed_text']]


In [ ]:
# @title API settings

# models approached by the OpenAI API
OpenAI_MODELS = [
    "gpt-5-mini-2025-08-07",
    "gpt-5-nano-2025-08-07",
    "gpt-4o-2024-08-06",
    "gpt-4o-mini-2024-07-18",
    "o4-mini-2025-04-16",
    "o3-mini-2025-01-31",
    "gpt-3.5-turbo-0125"]

# OpenAI API-key
OPENAI_KEY = "api_key_placeholder"

# models approached by the GROQ API
GROQ_MODELS = [
    "llama-3.3-70b-versatile",
    "deepseek-r1-distill-llama-70b"]

# GROQ API-key
GROQ_KEY = "api_key_placeholder"

# GROQ URL
GROQ_URL ="https://api.groq.com/openai/v1"

# Prepare content

In [ ]:
# @title Create configurations

### I have to research more extensively which dimensions I want to apply here!

# define models (as used by Cummins)
MODELS = [
    "gpt-5-mini-2025-08-07",
    "gpt-5-nano-2025-08-07",
    "gpt-4o-2024-08-06",
    "gpt-4o-mini-2024-07-18",
    "o4-mini-2025-04-16",
    "o3-mini-2025-01-31",
    "gpt-3.5-turbo-0125",  # models below this are called with Groq
    "llama-3.3-70b-versatile",
    "deepseek-r1-distill-llama-70b",
]

# define temperature parameters (as used by Cummins)
TEMPS = [0.0, 0.5, 1.0, 1.5]
# define reasoning parameters (as used by Cummins)
REASONING = ["low", "high"]
# Prompt style (CoT)
PROMPT_STYLES = [
    "Direct_Answer",
    "CoT_3_Steps",
    "CoT_5_Steps"
]
# Pipeline differences
# Iterations
# Amount of parsed text?
# ...

# generates every single combination
raw_grid = list(itertools.product(MODELS, TEMPS, REASONING, PROMPT_STYLES)) # maybye additional dimensions
grid_df = pd.DataFrame(raw_grid, columns=["model", "temperature", "reasoning_effort", "prompt_content"])

# conditional deletion weather a model has a temperature or reasoning paramter
reasoning_models_set = {"o4-mini-2025-04-16", "gpt-5-mini-2025-08-07",
                        "o3-mini-2025-01-31", "gpt-5-nano-2025-08-07"}
grid_df["reasoning_effort"] = np.where(
    ~grid_df["model"].isin(reasoning_models_set), np.nan, grid_df["reasoning_effort"]
)
grid_df["temperature"] = np.where(
    grid_df["model"].isin(reasoning_models_set), np.nan, grid_df["temperature"]
)

# drop duplicates
experiment_config = grid_df.drop_duplicates(ignore_index=True)

print(f"Total valid parameter combos: {len(experiment_config)}")
display(experiment_config)

print("\nConfiguration table completed!")

# API

In [ ]:
# @title Define API-calls

# --- Key Retrieval (Adapted from your setup) ---
try:

    # retrieve keys
    openai_api_key = OPENAI_KEY
    groq_api_key = GROQ_KEY

    # initialize the OpenAI client
    openai_client = OpenAI(api_key=openai_api_key)

    # initialize the GROQ client
    groq_client = OpenAI(
        api_key=groq_api_key,
        groq_url=GROQ_URL
    )

    # message after initialization
    print("API Clients initialized successfully.")

except Exception as e:
    # error message
    print(f"ERROR during client initialization: {e}")
    # raise error
    raise RuntimeError(f"Client initialization failed: {e}")

for config in experiment_config:

    model_name = config['model'] # dimension

    raw_temp = config.get('temperature') # dimension
    temp = raw_temp if pd.notna(raw_temp) else None

    raw_reasoning_effort = config.get('reasoning_effort')
    reasoning_effort = raw_reasoning_effort if pd.notna(raw_reasoning_effort) else None ### The API might not be able to process this argument!

    prompt_content = config['prompt_content'] ### The API might not be able to process this argument!

    # message of current model
    print(f"\nProcessing model: {model_name}")

    # client selection
    if model_name in GROQ_MODELS:
        client_to_use = groq_client
        print(f"  --> Selected GROQ client.")
    else:
        client_to_use = openai_client
        print(f"  --> Selected OPENAI client.")

    # basic content of the API-call
    completion_kwargs = {
        'model': model_name,
        'messages': [
            {"role": "user", "content": prompt_content} ### What shouuld be the user content? See in Jamie's Colab?!
        ],
    }

    # Add temperature if it is defined
    if temp is not None:
        completion_kwargs['temperature'] = temp

    # Conditionally add reasoning_effort (using a hypothetical parameter name)
    if reasoning_effort is not None:
        completion_kwargs['custom_reasoning_level'] = reasoning_effort
    #
    try:
        # **completion_kwargs unpacks the dictionary into arguments
        response = client_to_use.chat.completions.create(**completion_kwargs)

        # capture response
        generated_text = response.choices[0].message.content
        # status message
        print("\nAPI call completed!")
        # content message
        print(f"Response from {model_name}: {generated_text}")

    except Exception as e:
        # error message
        print(f"\nAPI Call Failed for {model_name}. Error: {e}")

print("\n API-calls complete")

In [ ]:
# @title execution

# Dataset

In [ ]:
# @title Attach results